# Práctica guiada de Minería de Datos para Ingeniería de Videojuegos

## Caso práctico
Trabajas en el equipo de **Game Analytics** de un estudio que gestiona un videojuego online multijugador.
Dispones de un dataset con telemetría agregada de jugadores y debes analizarlo para encontrar patrones de comportamiento y construir un modelo sencillo de **churn**.

## Qué se trabaja en esta práctica
- Comprensión del problema
- Calidad de datos
- Limpieza y transformación
- Exploración visual
- Reducción de dimensionalidad con **PCA**
- Modelo base de clasificación

## Instrucciones
- En algunas celdas ya tienes parte del código hecha.
- Tendrás que completar las zonas marcadas con `# TODO`.
- No se trata solo de ejecutar: **interpreta** lo que ves.


## 1. Importación de librerías

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# TODO: importa seaborn con el alias habitual

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

plt.figure(figsize=(8,5))
pd.set_option('display.max_columns', None)


## 2. Carga del dataset

In [ ]:
# El archivo debe estar en la misma carpeta que este notebook.

ruta = 'videojuegos_online_telemetria.csv'

# TODO: carga el CSV en un DataFrame llamado df


# TODO: muestra las 5 primeras filas


### Preguntas rápidas
1. ¿Qué representa cada fila?
2. ¿Qué variable parece ser la variable objetivo?
3. ¿Qué decisiones de producto o diseño podría apoyar este análisis?


## 3. Comprensión inicial del dataset

In [ ]:
print('Dimensiones del dataset:', df.shape)
print('\nTipos de datos:')
display(df.dtypes)

print('\nResumen estadístico de variables numéricas:')
display(df.describe())


In [ ]:
# TODO: muestra cuántos jugadores hay por plataforma
# Pista: value_counts()


# TODO: muestra cuántos jugadores hay por región


## 4. Calidad del dato

In [ ]:
print('Valores nulos por columna:')
display(df.isna().sum().sort_values(ascending=False))

print('Filas duplicadas:', df.duplicated().sum())


In [ ]:
print('Valores únicos de Region:')
print(df['Region'].unique())

print('\nValores únicos de Platform:')
print(df['Platform'].unique())


### Pregunta
¿Detectas inconsistencias en variables categóricas? Escríbelas aquí:

- 
- 


## 5. Limpieza de datos

In [ ]:
df_clean = df.copy()

# Eliminamos duplicados
df_clean = df_clean.drop_duplicates()

# TODO: corrige categorías inconsistentes
# Ejemplos esperados:
# - 'pc' debe pasar a 'PC'
# - 'Europe' debe pasar a 'EU'


print(df_clean.shape)


In [ ]:
# Imputación sencilla de nulos en variables numéricas con la mediana
num_cols = df_clean.select_dtypes(include=[np.number]).columns.tolist()

for col in num_cols:
    df_clean[col] = df_clean[col].fillna(df_clean[col].median())

# TODO: comprueba que ya no queden nulos



### Reflexión
¿Por qué tiene sentido usar la **mediana** en vez de la media en algunas variables de telemetría?

## 6. Exploración visual

In [ ]:
plt.figure(figsize=(6,4))
df_clean['Churn30D'].value_counts().sort_index().plot(kind='bar')
plt.title('Distribución de churn a 30 días')
plt.xlabel('Churn30D')
plt.ylabel('Número de jugadores')
plt.show()


In [ ]:
plt.figure(figsize=(7,4))
# TODO: crea un boxplot de SessionsPerWeek separado por Churn30D
# Pista: sns.boxplot(data=..., x='Churn30D', y='SessionsPerWeek')

plt.title('Sesiones por semana según churn')
plt.show()


In [ ]:
plt.figure(figsize=(7,4))
sns.boxplot(data=df_clean, x='Churn30D', y='InGamePurchase')
plt.title('Gasto in-game según churn')
plt.show()


In [ ]:
# TODO: calcula el churn medio por plataforma
# Pista: groupby + mean



### Preguntas de interpretación
1. ¿Los jugadores con menos sesiones parecen abandonar más?
2. ¿Parece existir relación entre gasto y retención?
3. ¿Ves alguna señal útil para un equipo de LiveOps?


## 7. Preparación para modelado

In [ ]:
y = df_clean['Churn30D']

# Quitamos la variable objetivo y el identificador
X = df_clean.drop(columns=['Churn30D', 'PlayerID'])

# Convertimos variables categóricas a numéricas
X = pd.get_dummies(X, drop_first=True)

print('Dimensiones de X:', X.shape)
display(X.head())


## 8. Estandarización y PCA

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca_full = PCA()
X_pca_full = pca_full.fit_transform(X_scaled)

var_exp = pca_full.explained_variance_ratio_
var_exp_acum = np.cumsum(var_exp)

display(var_exp[:10])
display(var_exp_acum[:10])


In [ ]:
plt.figure(figsize=(8,4))
plt.plot(range(1, len(var_exp_acum)+1), var_exp_acum, marker='o')
plt.axhline(y=0.80, linestyle='--')
plt.axhline(y=0.90, linestyle='--')
plt.xlabel('Número de componentes')
plt.ylabel('Varianza explicada acumulada')
plt.title('PCA - Varianza explicada acumulada')
plt.show()


In [ ]:
# TODO: crea un PCA de 2 componentes y transforma X_scaled


# TODO: crea un DataFrame llamado pca_df con columnas PC1 y PC2
# Añade también la columna Churn30D


In [ ]:
plt.figure(figsize=(8,5))
# TODO: crea un scatterplot de PC1 vs PC2 coloreado por Churn30D
# Pista: sns.scatterplot(...)

plt.title('Jugadores proyectados en 2 componentes principales')
plt.show()


### Preguntas de interpretación
1. ¿Se aprecia alguna separación visual entre perfiles?
2. ¿Qué ventaja aporta PCA en un problema con muchas variables?
3. ¿Qué inconveniente tiene usar PCA desde el punto de vista de interpretabilidad?


## 9. Modelo base de clasificación

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.25, random_state=42, stratify=y
)

print(X_train.shape, X_test.shape)


In [ ]:
modelo = LogisticRegression(max_iter=2000)
modelo.fit(X_train, y_train)

y_pred = modelo.predict(X_test)


In [ ]:
print('Accuracy:', accuracy_score(y_test, y_pred))
print('\nClassification report:\n')
print(classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred)
print('Matriz de confusión:')
print(cm)


In [ ]:
# TODO: representa la matriz de confusión con un heatmap
# Pista: sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')



## 10. Conclusiones finales

Responde brevemente:

1. ¿Qué variables parecen más relacionadas con el churn?
2. ¿Qué acciones propondrías al equipo de diseño, retención o monetización?
3. ¿Qué has aprendido sobre preparación de datos y PCA en un contexto de videojuegos?
